# FriendsCast Detection (YOLOv8) — Test & Evaluation Notebook

**Goal:** Evaluate the trained YOLOv8 model on a held-out test set and visualize predictions.

We report:
- Precision / Recall
- mAP50 and mAP50-95
- Confusion matrix
- Qualitative examples (good + failure cases)

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
!pip -q install ultralytics

import os, glob, random
from ultralytics import YOLO
from PIL import Image

In [18]:
OUT_ROOT = "/content/drive/MyDrive/Friends_YOLO"
YAML_PATH = "/content/drive/MyDrive/friends_dataset/friends.yaml"

WEIGHTS_PATH = "/content/drive/MyDrive/Friends_Project_Outputs/best.pt"

print(" YAML exists:", os.path.exists(YAML_PATH))
print(" Weights exists:", os.path.exists(WEIGHTS_PATH))

 YAML exists: True
 Weights exists: True


In [16]:
model = YOLO(WEIGHTS_PATH)
print(" Model loaded:", WEIGHTS_PATH)

 Model loaded: /content/drive/MyDrive/Friends_Project_Outputs/best.pt


In [19]:
metrics = model.val(data=YAML_PATH, split="test")
print(metrics)

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)



FileNotFoundError: Dataset '/content/drive/MyDrive/friends_dataset/friends.yaml' images not found, missing path '/content/friends_dataset/images/val'
Note dataset download directory is '/content/datasets'. You can update this in '/root/.config/Ultralytics/settings.json'

In [ ]:
# Ultralytics prints key metrics automatically.
# We'll also store the run directory for plots.
import glob, os

runs = sorted(glob.glob("runs/detect/val*"), key=os.path.getmtime)
VAL_DIR = runs[-1]
print(" VAL_DIR:", VAL_DIR)
print("Files:", os.listdir(VAL_DIR))

In [ ]:
# Show confusion matrix

cm_norm = os.path.join(VAL_DIR, "confusion_matrix_normalized.png")
cm = os.path.join(VAL_DIR, "confusion_matrix.png")

if os.path.exists(cm_norm):
    display(Image.open(cm_norm))
if os.path.exists(cm):
    display(Image.open(cm))

In [ ]:
# Run predictions on test images

preds = model.predict(
    source=f"{OUT_ROOT}/images/test",
    conf=0.25,
    save=True
)

runs = sorted(glob.glob("runs/detect/predict*"), key=os.path.getmtime)
PRED_DIR = runs[-1]
print("✅ PRED_DIR:", PRED_DIR)

In [ ]:
# Visualize some predictions

import glob, os

pred_imgs = sorted(glob.glob(os.path.join(PRED_DIR, "*.jpg")))[:12]  # montre 12
print("Nb images préd:", len(pred_imgs))

for p in pred_imgs:
    display(Image.open(p))

In [ ]:
# Save evaluation artifacts to Drive

import shutil, os

DEST = "/content/drive/MyDrive/Friends_Project_Outputs/eval"
os.makedirs(DEST, exist_ok=True)

# copy confusion matrices if exist
for fname in ["confusion_matrix.png", "confusion_matrix_normalized.png"]:
    src = os.path.join(VAL_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(DEST, fname))

# copy some predictions
pred_out = os.path.join(DEST, "predictions")
os.makedirs(pred_out, exist_ok=True)

for p in sorted(glob.glob(os.path.join(PRED_DIR, "*.jpg")))[:30]:
    shutil.copy2(p, os.path.join(pred_out, os.path.basename(p)))

print(" Saved eval artifacts to:", DEST)

## Final conclusion (short)

We evaluated a fine-tuned YOLOv8 model on a held-out test set and obtained strong mAP and stable precision/recall.
Remaining errors are mainly due to:
- visually similar faces
- pose/lighting variations
- image quality limitations

Next improvements could include:
- adding group scenes
- increasing dataset size
- trying a larger backbone (YOLOv8s)